In [ ]:
! pip install pypdf

In [2]:
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph,START,END
from langchain_huggingface import HuggingFaceEmbeddings
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.tools import tool
from typing import Annotated,TypedDict
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage,HumanMessage
from langgraph.prebuilt import ToolNode,tools_condition

/home/ai-with-rohan/Desktop/LangGraph/myvenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

True

In [4]:
llm = ChatGroq(model='openai/gpt-oss-20b')

> ## 1.Load the Documents

In [ ]:
loader = PyPDFLoader('LLM.pdf')
doc = loader.load()

In [ ]:
len(doc)

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size= 1000,chunk_overlap=200)
chunks = splitter.split_documents(doc)

In [ ]:
len(chunks)

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2',
    model_kwargs={'device': 'cpu'},  # or 'cuda' if GPU available
    encode_kwargs={'normalize_embeddings': False}  # Set to True for cosinesimilarity
)

In [ ]:
vector_store = FAISS.from_documents(chunks,embeddings)
vector_store

In [ ]:
retriever = vector_store.as_retriever(search_type='similarity',search_kwargs={'k':'4'})

In [3]:
@tool 
def rag_tool(query):
    """
    Retrieve relevant documents from the pdf document.
    Use this tool when the user asks the factual/conceptual questions
    that might be answered from the stored documents.
    """
    
    result = retriever.invoke(query)
    context = [doc.page_content for doc in result]
    metadata= [doc.metadata for doc in result]

    return{
        'query':query,
        'context':context,
        'metadata':metadata
    }

In [9]:
tools = [rag_tool]
llm_with_tools = llm.bind_tools(tools)

In [11]:
class ChatState(TypedDict):
    messages : Annotated[list[BaseMessage],add_messages]

In [12]:
def chat_node(state:ChatState):
    messages = state['messages']
    response = llm_with_tools.invoke(messages)
    return {'messages':[response]}

In [13]:
tool_node = ToolNode(tools)

In [15]:
graph = StateGraph(ChatState)

graph.add_node('chat_node',chat_node)
graph.add_node('tools',tool_node)

graph.add_edge(START,'chat_node')
graph.add_conditional_edges('chat_node',tools_condition)
graph.add_edge('tools','chat_node')
graph.add_edge('chat_node',END)

chatbot = graph.compile()

In [ ]:
chatbot

In [ ]:
result = chatbot.invoke(
    {
        'messages': [
            HumanMessage(
                content=(
                    'Using the pdf notes, explain how does sentence transformer work?'
                )
            )
        ]
    }
)

In [ ]:
print(result['messages'][-1].content)